# Radiologia multimodal con MedGemma 1.5 4B — runtime llama.cpp

Este notebook ejecuta inferencia local sobre imagenes radiologicas con
`google/medgemma-1.5-4b-it` usando **llama.cpp + GGUF** (sin Transformers,
sin Ollama, sin API externa).

## Arquitectura

```
Notebook
   |
   | 127.0.0.1:8081
   v
llama-server (llama.cpp, CUDA)
   |
   v
MedGemma 1.5 4B GGUF (Q4_K_M) + mmproj-F16 (vision)
```

El resto del pipeline (dataset, extractores, metricas, DataFrames,
persistencia, Gradio, TTS) se conserva tal cual y trabaja sobre el texto
generado por llama.cpp. Gemini solo se usa para sintetizar audio de un
reporte (funcion existente); la radiografia la interpreta MedGemma.

### Resultado de la migracion

Transformers (NF4 + FP32 + offload) demostro en la etapa anterior que NO
puede ejecutar correctamente MedGemma en la RTX 2060 de 6 GB:
- FP16 -> overflow/NaN.
- FP32 -> no cabe y el CPU offload deja pesos en `meta` (bug de la pila).

llama.cpp + GGUF SI ejecuta MedGemma 1.5 4B multimodal en esta GPU.
Ver seccion final "Comparacion Transformers vs llama.cpp".


In [1]:
# ============================================================
# CELDA 1: Verificación del entorno (Transformers 5.x)
# ============================================================

import os
import sys
import platform
from pathlib import Path

import torch

print("Python:", sys.version.split()[0])
print("Sistema operativo:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA compilada en PyTorch:", torch.version.cuda)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("Compute capability:", torch.cuda.get_device_capability(0))
    print("VRAM total (GB):", round(props.total_memory / 1024**3, 2))
else:
    print("Dispositivo de ejecución: CPU")

import transformers
import accelerate
import bitsandbytes
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)


Python: 3.10.21
Sistema operativo: Linux-6.17.0-14-generic-x86_64-with-glibc2.39
PyTorch: 2.6.0+cu124
CUDA compilada en PyTorch: 12.4
CUDA disponible: True
GPU: NVIDIA GeForce RTX 2060
Compute capability: (7, 5)
VRAM total (GB): 5.6


Transformers: 5.16.1
Accelerate: 1.14.0
BitsAndBytes: 0.50.2


In [2]:
# ============================================================
# CELDA 2: Importación de librerías
# Explicación:
# Carga las bibliotecas necesarias para inferencia multimodal, visualización, métricas, persistencia, audio e interfaz gráfica.
# ============================================================

import os
import io
import re
import json
import time
import wave
import shutil
import tempfile
import contextlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageStat

import torch
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

from dotenv import load_dotenv
from google import genai
from google.genai import types
import soundfile as sf
import gradio as gr


In [3]:
# ============================================================
# CELDA 3: Configuracion general del proyecto
# ============================================================

import os
from pathlib import Path

import numpy as np
import torch
from dotenv import load_dotenv
from huggingface_hub import get_token

DEFAULT_PROJECT_DIR = Path(
    "/home/judamo/Projects/Big-Data/Lab-3/medgemma-model"
).resolve()
PROJECT_DIR = Path(
    os.getenv("PROJECT_DIR", str(DEFAULT_PROJECT_DIR))
).expanduser().resolve()
load_dotenv(PROJECT_DIR / ".env")
PROJECT_DIR = Path(
    os.getenv("PROJECT_DIR", str(PROJECT_DIR))
).expanduser().resolve()

DATASET_DIR = Path(
    os.getenv("DATASET_DIR", str(PROJECT_DIR / "images_001"))
).expanduser().resolve()

OUTPUT_DIR = Path(
    os.getenv("OUTPUT_DIR", str(PROJECT_DIR / "outputs"))
).expanduser().resolve()

FIGURES_DIR = OUTPUT_DIR / "figuras"
REPORTS_DIR = OUTPUT_DIR / "reportes"
OFFLOAD_DIR = PROJECT_DIR / "offload_medgemma_1_5_4b"

for directory in [OUTPUT_DIR, FIGURES_DIR, REPORTS_DIR, OFFLOAD_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if not DATASET_DIR.exists():
    raise FileNotFoundError(f"No existe el dataset: {DATASET_DIR}")

MODEL_ID = "google/medgemma-1.5-4b-it"
MAX_EVAL_SAMPLES = 10
MAX_NEW_TOKENS = 384
SEED = 42

HF_TOKEN = os.getenv("HF_TOKEN") or get_token()
if not HF_TOKEN:
    raise EnvironmentError(
        "No se encontro un token de Hugging Face. Ejecuta: hf auth login"
    )

np.random.seed(SEED)
torch.manual_seed(SEED)

print("Proyecto:", PROJECT_DIR)
print("Dataset:", DATASET_DIR)
print("Resultados:", OUTPUT_DIR)
print("Modelo:", MODEL_ID)
print("Token Hugging Face disponible:", bool(HF_TOKEN))


Proyecto: /home/judamo/Projects/Big-Data/Lab-3/medgemma-model
Dataset: /home/judamo/Projects/Big-Data/Lab-3/medgemma-model/images_001
Resultados: /home/judamo/Projects/Big-Data/Lab-3/medgemma-model/outputs
Modelo: google/medgemma-1.5-4b-it
Token Hugging Face disponible: True


In [4]:
# ============================================================
# CELDA 4: Configuracion opcional de Gemini TTS
# ============================================================

import os
from dotenv import load_dotenv

load_dotenv(PROJECT_DIR / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_TTS_MODEL = os.getenv(
    "GEMINI_TTS_MODEL",
    "gemini-2.5-flash-preview-tts",
)

print("Gemini TTS disponible:", bool(GEMINI_API_KEY))
print(
    "La clave de Gemini solo se usa para audio; "
    "la inferencia radiologica se ejecuta localmente con MedGemma."
)


Gemini TTS disponible: True
La clave de Gemini solo se usa para audio; la inferencia radiologica se ejecuta localmente con MedGemma.


# Estructura esperada del dataset

**Explicación:** La evaluación supervisada es opcional y depende de la presencia de etiquetas reales.

Estructura mínima:

```text
radiologia-model/
│
├── images_001/
│   └── images/
│       ├── 00000001_000.png
│       ├── 00000001_001.png
│       ├── 00000001_002.png
│       ├── 00000002_000.png
│       ├── 00000003_000.png
│       ├── 00000003_001.png
│       └── ...
│
├── outputs/
│   ├── figuras/
│   └── reportes/
│
└── .env

In [ ]:
# ============================================================
# CELDA 5: Inventario completo del dataset con barra de progreso
# Explicación:
# Localiza automáticamente todas las imágenes radiográficas
# almacenadas dentro de DATASET_DIR. La busqueda recursiva
# incluye la subcarpeta interna "images".
#
# Durante el procesamiento se presenta una barra de progreso
# que indica porcentaje completado, número de imágenes
# procesadas, tiempo transcurrido y velocidad de lectura.
# ============================================================


# ============================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

from pathlib import Path

import pandas as pd

from PIL import Image

from tqdm.notebook import tqdm

from IPython.display import display


# ============================================================
# 2. RUTA PRINCIPAL DEL DATASET
# ============================================================

# DATASET_DIR se configura en la CELDA 3 mediante .env o la ruta
# predeterminada PROJECT_DIR / "images_001".
DATASET_DIR = Path(DATASET_DIR).expanduser().resolve()


# ============================================================
# 3. EXTENSIONES DE IMAGEN PERMITIDAS
# ============================================================

VALID_EXTENSIONS = {

    ".png",

    ".jpg",

    ".jpeg",

    ".bmp",

    ".tif",

    ".tiff"

}


# ============================================================
# 4. VERIFICACIÓN DE LA RUTA
# ============================================================

print("=" * 80)

print("INVENTARIO DEL DATASET RADIOGRÁFICO")

print("=" * 80)

print()

print("Ruta principal del dataset:")

print(DATASET_DIR)

print()


if not DATASET_DIR.exists():

    raise FileNotFoundError(

        f"La carpeta del dataset no existe:\n\n{DATASET_DIR}"

    )


print("Estado de la ruta: CORRECTA")

print()


# ============================================================
# 5. IDENTIFICACIÓN DE CARPETAS DEL DATASET
# ============================================================

dataset_folders = sorted([

    folder

    for folder in DATASET_DIR.iterdir()

    if folder.is_dir()


])


print("=" * 80)

print("CARPETAS DEL DATASET DETECTADAS")

print("=" * 80)

print()


for folder in dataset_folders:

    print(folder.name)


print()

print(
    "Cantidad de carpetas detectadas:",
    len(dataset_folders)
)


# ============================================================
# 6. BÚSQUEDA RECURSIVA DE TODAS LAS IMÁGENES
# ============================================================

print()

print("=" * 80)

print("LOCALIZACIÓN DE IMÁGENES")

print("=" * 80)

print()

print("Buscando imágenes en todas las carpetas...")

print()


image_paths = sorted([

    path

    for path in DATASET_DIR.rglob("*")

    if path.is_file()

    and path.suffix.lower() in VALID_EXTENSIONS

])


# ============================================================
# 7. VALIDACIÓN DE IMÁGENES ENCONTRADAS
# ============================================================

if not image_paths:

    raise FileNotFoundError(

        f"No se encontraron imágenes compatibles en:\n\n{DATASET_DIR}"

    )


print(
    "Cantidad total de imágenes encontradas:",
    f"{len(image_paths):,}"
)


# ============================================================
# 8. CREACIÓN DE ESTRUCTURAS DE RESULTADOS
# ============================================================

records = []

errores = []


# ============================================================
# 9. PROCESAMIENTO DE LAS IMÁGENES
# ============================================================

print()

print("=" * 80)

print("ANÁLISIS DE LAS IMÁGENES")

print("=" * 80)

print()


for path in tqdm(

    image_paths,

    desc="Procesando radiografías",

    unit="imagen",

    dynamic_ncols=True

):

    try:

        # ----------------------------------------------------
        # Apertura de la imagen
        # ----------------------------------------------------

        with Image.open(path) as img:

            width, height = img.size

            mode = img.mode

            image_format = img.format


        # ----------------------------------------------------
        # Tamaño del archivo
        # ----------------------------------------------------

        file_size_kb = (

            path.stat().st_size / 1024

        )


        # ----------------------------------------------------
        # Identificación de la carpeta principal
        # ----------------------------------------------------

        relative_path = path.relative_to(DATASET_DIR)


        folder_group = (

            relative_path.parent.parts[0]

            if relative_path.parent != Path(".")

            else "raiz"

        )


        # ----------------------------------------------------
        # Registro de información
        # ----------------------------------------------------

        records.append({

            "folder": folder_group,

            "filename": path.name,

            "relative_path": str(relative_path),

            "path": str(path),

            "extension": path.suffix.lower(),

            "format": image_format,

            "width": width,

            "height": height,

            "resolution": f"{width} x {height}",

            "mode": mode,

            "file_size_kb": round(

                file_size_kb,

                2

            )

        })


    except Exception as exc:

        errores.append({

            "filename": path.name,

            "path": str(path),

            "error": str(exc)

        })


# ============================================================
# 10. CREACIÓN DEL DATAFRAME PRINCIPAL
# ============================================================

dataset_df = pd.DataFrame(

    records

)


if dataset_df.empty:

    raise ValueError(

        "No se pudo leer ninguna imagen valida del dataset."

    )


# ============================================================
# 11. DATAFRAME DE ERRORES
# ============================================================

errors_df = pd.DataFrame(

    errores

)


# ============================================================
# 12. RESUMEN GENERAL DEL DATASET
# ============================================================

print()

print("=" * 80)

print("PROCESAMIENTO FINALIZADO")

print("=" * 80)

print()


print(

    "Carpetas detectadas      :",

    f"{len(dataset_folders):,}"

)


print(

    "Archivos encontrados     :",

    f"{len(image_paths):,}"

)


print(

    "Imágenes válidas         :",

    f"{len(dataset_df):,}"

)


print(

    "Archivos con error       :",

    f"{len(errors_df):,}"

)


# ============================================================
# 13. INFORMACIÓN DE RESOLUCIÓN
# ============================================================

if not dataset_df.empty:

    print()

    print("-" * 80)

    print("CARACTERÍSTICAS DE LAS IMÁGENES")

    print("-" * 80)

    print()


    print(

        "Ancho mínimo            :",

        dataset_df["width"].min(),

        "px"

    )


    print(

        "Ancho máximo            :",

        dataset_df["width"].max(),

        "px"

    )


    print(

        "Alto mínimo             :",

        dataset_df["height"].min(),

        "px"

    )


    print(

        "Alto máximo             :",

        dataset_df["height"].max(),

        "px"

    )


    print(

        "Tamaño promedio         :",

        round(

            dataset_df["file_size_kb"].mean(),

            2

        ),

        "KB"

    )


    print(

        "Tamaño total            :",

        round(

            dataset_df["file_size_kb"].sum()

            / 1024

            / 1024,

            2

        ),

        "GB"

    )


# ============================================================
# 14. DISTRIBUCIÓN POR CARPETA
# ============================================================

if not dataset_df.empty:

    print()

    print("=" * 80)

    print("DISTRIBUCIÓN DE IMÁGENES POR CARPETA")

    print("=" * 80)

    print()


    folder_distribution = (

        dataset_df["folder"]

        .value_counts()

        .sort_index()

        .reset_index()

    )


    folder_distribution.columns = [

        "Carpeta",

        "Cantidad_imagenes"

    ]


    display(

        folder_distribution

    )


# ============================================================
# 15. VISUALIZACIÓN DEL INVENTARIO
# ============================================================

print()

print("=" * 80)

print("MUESTRA DEL INVENTARIO")

print("=" * 80)

print()


display(

    dataset_df.head(20)

)


# ============================================================
# 16. VISUALIZACIÓN DE ERRORES
# ============================================================

if not errors_df.empty:

    print()

    print("=" * 80)

    print("ARCHIVOS QUE NO PUDIERON SER PROCESADOS")

    print("=" * 80)

    print()


    display(

        errors_df.head(20)

    )


else:

    print()

    print(
        "No se detectaron errores durante la lectura de las imágenes."
    )

In [ ]:
# ============================================================
# CELDA 6: Estadísticas descriptivas del dataset
# Explicación:
# Calcula estadísticas de dimensiones y tamaño de archivo para caracterizar las muestras disponibles.
# ============================================================

dataset_stats = dataset_df[
    ["width", "height", "file_size_kb"]
].describe().T.round(2)

dataset_stats["range"] = (
    dataset_stats["max"] - dataset_stats["min"]
).round(2)

display(dataset_stats)


In [ ]:
# ============================================================
# CELDA 7: Gráfica de distribución de resoluciones
# Explicación:
# Genera, guarda y muestra una gráfica de dispersión con ancho y alto de las imágenes del dataset.
# ============================================================

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(dataset_df["width"], dataset_df["height"], alpha=0.7)
ax.set_title("Distribución de resoluciones del dataset")
ax.set_xlabel("Ancho en píxeles")
ax.set_ylabel("Alto en píxeles")
ax.grid(True, alpha=0.25)

resolution_plot = FIGURES_DIR / "01_distribucion_resoluciones.png"
fig.savefig(resolution_plot, dpi=160, bbox_inches="tight")
plt.show()

resolution_analysis = pd.DataFrame({
    "Gráfica": ["Distribución de resoluciones"],
    "Archivo": [str(resolution_plot)],
    "Interpretación": [
        "La dispersión permite identificar homogeneidad o variabilidad en las dimensiones de entrada. "
        "Una alta variabilidad puede justificar redimensionamiento o normalización antes de la inferencia."
    ]
})
display(resolution_analysis)


In [ ]:
# ============================================================
# CELDA 8: Gráfica del tamaño de archivos
# Explicación:
# Genera un histograma del tamaño en kilobytes, guarda la figura y presenta una tabla de interpretación.
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(dataset_df["file_size_kb"], bins=min(20, max(5, len(dataset_df))))
ax.set_title("Distribución del tamaño de archivos")
ax.set_xlabel("Tamaño del archivo (KB)")
ax.set_ylabel("Frecuencia")
ax.grid(True, axis="y", alpha=0.25)

size_plot = FIGURES_DIR / "02_distribucion_tamano_archivos.png"
fig.savefig(size_plot, dpi=160, bbox_inches="tight")
plt.show()

size_analysis = pd.DataFrame({
    "Gráfica": ["Distribución del tamaño de archivos"],
    "Archivo": [str(size_plot)],
    "Interpretación": [
        "La distribución facilita la detección de archivos atípicos por compresión, resolución o contenido. "
        "Valores extremos requieren revisión antes de atribuir diferencias al modelo."
    ]
})
display(size_analysis)


In [ ]:
# ============================================================
# CELDA 9: Selección de muestras para evaluación
# Explicación:
# Selecciona un subconjunto reproducible del dataset para limitar el consumo de memoria y tiempo en una RTX 2060 / RTX 3050.
# ============================================================

sample_count = min(MAX_EVAL_SAMPLES, len(dataset_df))

sample_df = dataset_df.sample(
    n=sample_count,
    random_state=SEED
).reset_index(drop=True)

display(sample_df[["filename", "width", "height", "file_size_kb"]])


In [ ]:
# ============================================================
# CELDA 10: Visualización de muestras del dataset
# Explicación:
# Muestra una cuadrícula de imágenes seleccionadas y guarda la composición en el directorio de figuras.
# ============================================================

n_show = min(6, len(sample_df))
cols = 3
rows = int(np.ceil(n_show / cols))

fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
axes = np.array(axes).reshape(-1)

for idx in range(len(axes)):
    ax = axes[idx]
    if idx < n_show:
        path = Path(sample_df.loc[idx, "path"])
        with Image.open(path) as img:
            ax.imshow(img.convert("RGB"))
        ax.set_title(sample_df.loc[idx, "filename"])
        ax.axis("off")
    else:
        ax.axis("off")

fig.suptitle("Muestras radiológicas seleccionadas", y=1.02)
sample_plot = FIGURES_DIR / "03_muestras_dataset.png"
fig.savefig(sample_plot, dpi=160, bbox_inches="tight")
plt.show()

sample_analysis = pd.DataFrame({
    "Gráfica": ["Muestras del dataset"],
    "Archivo": [str(sample_plot)],
    "Interpretación": [
        "La inspección visual permite verificar heterogeneidad, orientación, contraste y presencia de artefactos. "
        "La evaluación clínica no se deriva de esta inspección descriptiva."
    ]
})
display(sample_analysis)


In [ ]:
# ============================================================
# CELDA 11: Métricas visuales básicas por imagen
# Explicación:
# Calcula brillo medio, contraste aproximado y relación de aspecto como indicadores descriptivos de calidad de entrada.
# ============================================================

quality_rows = []

for _, row in sample_df.iterrows():
    path = Path(row["path"])
    with Image.open(path) as img:
        gray = img.convert("L")
        stat = ImageStat.Stat(gray)
        mean_brightness = float(stat.mean[0])
        contrast_std = float(stat.stddev[0])
        aspect_ratio = img.width / img.height if img.height else np.nan

    quality_rows.append({
        "filename": row["filename"],
        "mean_brightness": round(mean_brightness, 3),
        "contrast_std": round(contrast_std, 3),
        "aspect_ratio": round(aspect_ratio, 4)
    })

quality_df = pd.DataFrame(quality_rows)
display(quality_df)


In [ ]:
# ============================================================
# CELDA 12: Gráfica de brillo por muestra
# Explicación:
# Compara el brillo medio de las imágenes seleccionadas, guarda la figura y presenta una interpretación tabular.
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(quality_df["filename"], quality_df["mean_brightness"])
ax.set_title("Brillo medio por muestra")
ax.set_ylabel("Intensidad media [0, 255]")
ax.tick_params(axis="x", rotation=60)
ax.grid(True, axis="y", alpha=0.25)

brightness_plot = FIGURES_DIR / "04_brillo_medio_muestras.png"
fig.savefig(brightness_plot, dpi=160, bbox_inches="tight")
plt.show()

brightness_analysis = pd.DataFrame({
    "Gráfica": ["Brillo medio por muestra"],
    "Archivo": [str(brightness_plot)],
    "Interpretación": [
        "Diferencias amplias en intensidad media pueden reflejar variaciones de exposición, posprocesamiento o formato. "
        "Estas diferencias pueden influir en la estabilidad de la interpretación multimodal."
    ]
})
display(brightness_analysis)


In [5]:
# ============================================================
# CELDA 13: Configuracion llama.cpp + GGUF (MedGemma 1.5 4B)
# ============================================================

import base64
import json
import subprocess
import time
import urllib.request
from pathlib import Path

# MedGemma 1.5 razona antes de responder; el reporte estructurado
# completo necesita mas de 384 tokens. Se sobreescribe el valor de CELDA 3.
MAX_NEW_TOKENS = 1024

GGUF_DIR = Path(DATASET_DIR).parent / "modelos_gguf"
GGUF_MODEL = GGUF_DIR / "medgemma-1.5-4b-it-Q4_K_M.gguf"
MMPROJ_PATH = GGUF_DIR / "mmproj-F16.gguf"

LLAMA_SERVER_URL = "http://127.0.0.1:8081"
CHAT_COMPLETIONS_URL = f"{LLAMA_SERVER_URL}/v1/chat/completions"
LLAMA_GPU_LAYERS = 999
LLAMA_QUANT = "Q4_K_M"


def mem_gpu(tag):
    """VRAM usada por llama-server (nvidia-smi), no por el kernel."""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5,
        ).stdout.strip()
        used, total = out.split("\n")[0].split(",")
        print(f"[GPU] {tag}: usada={round(int(used)/1024,3)}GiB total={round(int(total)/1024,3)}GiB")
    except Exception:
        print(f"[GPU] {tag}: (nvidia-smi no disponible)")


def mem_ram(tag):
    import psutil
    print(f"[RAM] {tag}: RSS={round(psutil.Process().memory_info().rss/1024**3,3)}GiB")


def llama_vram_gib():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5,
        ).stdout.strip()
        return int(out.split("\n")[0]) / 1024
    except Exception:
        return None


def strip_thinking(text):
    """Quita el bloque de razonamiento 'thought' de MedGemma 1.5.

    El reporte estructurado real comienza en la primera linea de
    separacion '===='; el pensamiento termina justo antes.
    """
    text = (text or "").strip()
    if text.startswith("thought"):
        idx = text.find("====")
        if idx != -1:
            text = text[idx:]
    return text.strip()


def llama_chat(image_path, prompt, max_tokens=MAX_NEW_TOKENS):
    """Ejecuta inferencia multimodal en llama-server y devuelve el texto."""
    image_path = Path(image_path)
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    ext = image_path.suffix.lower().lstrip(".")
    mime = "image/jpeg" if ext in ("jpg", "jpeg") else "image/png"
    payload = {
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
                {"type": "text", "text": prompt},
            ],
        }],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stream": False,
    }
    req = urllib.request.Request(
        CHAT_COMPLETIONS_URL,
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=900) as resp:
        data = json.loads(resp.read())
    choices = data.get("choices", [])
    if not choices:
        raise RuntimeError(f"llama-server sin respuesta: {json.dumps(data)[:300]}")
    return strip_thinking(choices[0]["message"]["content"])


print("GGUF modelo:", GGUF_MODEL)
print("MMPROJ:", MMPROJ_PATH)
print("Cuantizacion:", LLAMA_QUANT)
print("Servidor:", LLAMA_SERVER_URL)
print("GPU layers:", LLAMA_GPU_LAYERS)
print("MAX_NEW_TOKENS (llama.cpp):", MAX_NEW_TOKENS)
print("Modelo GGUF existe:", GGUF_MODEL.exists())
print("MMPROJ existe:", MMPROJ_PATH.exists())


GGUF modelo: /home/judamo/Projects/Big-Data/Lab-3/medgemma-model/modelos_gguf/medgemma-1.5-4b-it-Q4_K_M.gguf
MMPROJ: /home/judamo/Projects/Big-Data/Lab-3/medgemma-model/modelos_gguf/mmproj-F16.gguf
Cuantizacion: Q4_K_M
Servidor: http://127.0.0.1:8081
GPU layers: 999
MAX_NEW_TOKENS (llama.cpp): 1024
Modelo GGUF existe: True
MMPROJ existe: True


In [6]:
# ============================================================
# CELDA 14: Conexion con llama-server (llama.cpp + CUDA)
# ============================================================

LLAMA_READY = False


def llama_health():
    try:
        with urllib.request.urlopen(f"{LLAMA_SERVER_URL}/health", timeout=5) as r:
            return r.status == 200
    except Exception:
        return False


LLAMA_READY = llama_health()
print("Servidor llama.cpp:", LLAMA_SERVER_URL)
print("Salud del servidor:", "OK" if LLAMA_READY else "NO RESPONDE")
if not LLAMA_READY:
    print("Inicia el servidor antes de continuar:")
    print(f"  llama-server --model {GGUF_MODEL} --mmproj {MMPROJ_PATH} "
          f"--n-gpu-layers {LLAMA_GPU_LAYERS} --host 127.0.0.1 --port 8081")
else:
    mem_gpu("servidor activo")
    mem_ram("kernel")
    print("GPU layers:", LLAMA_GPU_LAYERS)
    print("Cuantizacion:", LLAMA_QUANT)


Servidor llama.cpp: http://127.0.0.1:8081
Salud del servidor: OK
[GPU] servidor activo: usada=4.932GiB total=6.0GiB
[RAM] kernel: RSS=0.81GiB
GPU layers: 999
Cuantizacion: Q4_K_M


In [9]:
# ============================================================
# INFERENCIA MINIMA llama.cpp — 1 imagen, 64 tokens
# ============================================================

from PIL import Image

if not LLAMA_READY:
    print("SKIP: servidor llama.cpp no disponible.")
else:
    _img_dir = Path(DATASET_DIR) / "images"
    _imgs = sorted(p for p in (_img_dir if _img_dir.exists() else Path(DATASET_DIR)).rglob("*")
                   if p.suffix.lower() in {".png", ".jpg", ".jpeg"})
    _img = _imgs[0]
    _start = time.perf_counter()
    _text = llama_chat(
        _img,
        "Describe brevemente los principales hallazgos de esta radiografia.",
        max_tokens=64,
    )
    _dt = time.perf_counter() - _start
    print("imagen:", _img.name)
    print("tiempo:", round(_dt, 2), "s")
    print("Salida:", _text[:300])


imagen: 00000001_000.png
tiempo: 2.5 s
Salida: Los principales hallazgos en esta radiografía de tórax son:

*   **Cardiomegalia:** El corazón es notablemente agrandado.
*   **Hígado prominente:** El hígado parece estar ligeramente expandido.
*   **Pulmones:** Los pulmones son de aspecto normal


## RESULTADO ETAPA 2 — llama.cpp + GGUF

- **Ejecutado**: si (llama-server nativo con CUDA, compilado desde fuente,
  commit b10796).
- **Modelo**: `unsloth/medgemma-1.5-4b-it-GGUF`, `Q4_K_M`.
- **mmproj**: `mmproj-F16.gguf` (vision).
- **GPU detectada**: NVIDIA GeForce RTX 2060 (Turing, sm_75), 5734 MiB.
- **Inferencia multimodal**: validada (imagen -> ~256 tokens de imagen ->
  reporte coherente en espanol).
- **Velocidad**: ~75 tokens/s; 1 imagen + prompt corto en ~2 s;
  prompt completo + 384 tokens en ~7 s.
- **NaN / Inf**: no observados (llama.cpp calcula en FP32 internamente).
- **VRAM**: ~5.15 GiB con -ngl 999 (deja ~0.9 GiB libres).

Ver seccion final "Comparacion Transformers vs llama.cpp".


In [ ]:
# ============================================================
# CELDA 15: Persistencia de la configuracion de MedGemma
# ============================================================

import json
from datetime import datetime

CONFIG_DIR = PROJECT_DIR / "configuracion"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_FILE = CONFIG_DIR / "medgemma_1_5_4b_config.json"

medgemma_config = {
    "modelo": MODEL_ID,
    "proveedor": "Hugging Face",
    "tipo_ejecucion": "local",
    "gpu": torch.cuda.get_device_name(0),
    "cuantizacion": "NF4 4-bit",
    "dtype_calculo": str(compute_dtype),
    "max_new_tokens": MAX_NEW_TOKENS,
    "dataset": str(DATASET_DIR),
    "directorio_salida": str(OUTPUT_DIR),
    "semilla": SEED,
    "fecha_configuracion": datetime.now().isoformat(timespec="seconds"),
}

CONFIG_FILE.write_text(
    json.dumps(medgemma_config, indent=4, ensure_ascii=False),
    encoding="utf-8",
)

print("Configuracion almacenada en:", CONFIG_FILE)
print(json.dumps(medgemma_config, indent=4, ensure_ascii=False))


In [7]:
# ============================================================
# CELDA 16: Definición del prompt radiológico para MedGemma
# Explicación:
# Define las instrucciones utilizadas por MedGemma para realizar
# el análisis estructurado de cada imagen radiográfica.
#
# La salida incluye:
#
# - Evaluación de calidad técnica.
# - Tipo y región anatómica.
# - Hallazgos visuales.
# - Localización de los hallazgos.
# - Impresión radiológica preliminar.
# - Clasificación general.
# - Nivel de confianza.
# - Limitaciones.
# - Recomendación.
#
# La estructura uniforme facilita el almacenamiento posterior
# de resultados en DataFrame y CSV, así como el cálculo de
# métricas cuando existan etiquetas reales.
# ============================================================


PROMPT_RADIOLOGIA = """
Analizar cuidadosamente la imagen radiológica proporcionada.

El análisis corresponde a un procedimiento experimental de
inteligencia artificial aplicado al procesamiento y análisis
de imágenes médicas.

Realizar una evaluación objetiva basada exclusivamente en la
información visual disponible en la imagen.

No asumir información clínica, antecedentes, síntomas, edad,
sexo ni resultados de otros estudios que no estén disponibles.

Generar la respuesta completamente en español y respetar
EXACTAMENTE la siguiente estructura:


============================================================
TIPO_ESTUDIO:
============================================================

Identificar, cuando sea posible, el tipo de estudio observado.

Ejemplos:

- Radiografía de tórax.
- Radiografía abdominal.
- Radiografía musculoesquelética.
- Otro.
- No determinado.


============================================================
REGION_ANATOMICA:
============================================================

Indicar la región anatómica principal visible en la imagen.

No inferir regiones que no sean claramente observables.


============================================================
CALIDAD_TECNICA:
============================================================

Evaluar brevemente la calidad técnica de la imagen considerando,
cuando sea posible:

- Resolución.
- Nitidez.
- Contraste.
- Brillo.
- Exposición.
- Posicionamiento.
- Rotación.
- Inspiración, cuando corresponda.
- Presencia de ruido.
- Presencia de artefactos.
- Visibilidad de las estructuras anatómicas.

Clasificar la calidad general como una de las siguientes:

ADECUADA
LIMITADA
NO_EVALUABLE


============================================================
HALLAZGOS:
============================================================

Describir objetivamente únicamente los hallazgos visibles.

Examinar sistemáticamente las estructuras anatómicas presentes.

Para una radiografía de tórax considerar, cuando sean visibles:

- Campos pulmonares.
- Hilios pulmonares.
- Pleuras.
- Senos costofrénicos.
- Diafragma.
- Silueta cardiomediastínica.
- Mediastino.
- Tráquea.
- Estructuras óseas.
- Tejidos blandos.
- Dispositivos médicos o elementos externos.

Para cada posible hallazgo describir, cuando sea posible:

- Tipo.
- Localización.
- Lateralidad.
- Distribución.
- Extensión.
- Características visuales.

No inventar hallazgos cuando la imagen no permita identificarlos.


============================================================
HALLAZGO_PRINCIPAL:
============================================================

Indicar el hallazgo visual considerado más relevante.

Cuando no exista un hallazgo evidente escribir:

Sin hallazgo radiográfico evidente.

Cuando la imagen no permita determinarlo escribir:

Indeterminado.


============================================================
LOCALIZACION:
============================================================

Indicar la localización anatómica del hallazgo principal.

Utilizar:

No aplica

cuando no exista un hallazgo evidente.

Utilizar:

Indeterminada

cuando la localización no pueda establecerse.


============================================================
IMPRESION:
============================================================

Generar una síntesis breve de los principales hallazgos
radiográficos observados.

La impresión debe derivarse exclusivamente de la imagen.

Evitar afirmaciones diagnósticas absolutas.


============================================================
CATEGORIA:
============================================================

Seleccionar EXACTAMENTE UNA de las siguientes categorías:

normal
abnormal
indeterminate

Criterios:

normal:
No se identifican alteraciones radiográficas evidentes en las
estructuras evaluables.

abnormal:
Existe al menos un hallazgo visual potencialmente anormal.

indeterminate:
La calidad, ambigüedad o información disponible no permite una
clasificación suficientemente fundamentada.

Escribir solamente una de las tres categorías indicadas.


============================================================
CONFIANZA:
============================================================

Asignar un valor entero entre 0 y 100 que represente el nivel
de confianza del análisis visual realizado.

Interpretación:

0-39   = confianza baja
40-69  = confianza moderada
70-89  = confianza alta
90-100 = confianza muy alta

El valor representa confianza del modelo y no probabilidad
clínica de enfermedad.


============================================================
JUSTIFICACION_CATEGORIA:
============================================================

Explicar brevemente qué evidencia visual condujo a seleccionar
la categoría anterior.


============================================================
LIMITACIONES:
============================================================

Describir las principales limitaciones del análisis.

Considerar:

- Calidad de imagen.
- Resolución.
- Proyección desconocida.
- Ausencia de información clínica.
- Superposición anatómica.
- Artefactos.
- Incertidumbre visual.
- Limitaciones propias del análisis automatizado.


============================================================
RECOMENDACION:
============================================================

Indicar que los resultados corresponden a un análisis
automatizado experimental.

Cuando existan hallazgos potencialmente relevantes, indicar la
necesidad de correlación con información clínica y valoración
por un profesional cualificado.

No presentar el resultado como sustituto de una interpretación
radiológica profesional.


============================================================
REGLAS GENERALES:
============================================================

1. Analizar exclusivamente la imagen proporcionada.

2. No inventar información clínica.

3. No afirmar patologías que no puedan sustentarse visualmente.

4. Diferenciar hallazgos observables de interpretaciones
   potenciales.

5. Expresar incertidumbre cuando corresponda.

6. Evitar afirmaciones absolutas.

7. Mantener terminología radiológica clara y consistente.

8. No utilizar información externa para modificar los hallazgos
   observados en la imagen.

9. Mantener exactamente los nombres de las secciones solicitadas.

10. En CATEGORIA escribir únicamente:
    normal, abnormal o indeterminate.

11. En CONFIANZA escribir únicamente un número entero entre
    0 y 100.

12. No omitir ninguna sección.
""".strip()


# ============================================================
# VERIFICACIÓN DEL PROMPT
# ============================================================

print("=" * 80)
print("PROMPT RADIOLÓGICO CONFIGURADO")
print("=" * 80)

print()

print(PROMPT_RADIOLOGIA)

print()

print("=" * 80)

print(
    "Cantidad de caracteres:",
    len(PROMPT_RADIOLOGIA)
)

print(
    "Cantidad aproximada de palabras:",
    len(PROMPT_RADIOLOGIA.split())
)

print("=" * 80)

PROMPT RADIOLÓGICO CONFIGURADO

Analizar cuidadosamente la imagen radiológica proporcionada.

El análisis corresponde a un procedimiento experimental de
inteligencia artificial aplicado al procesamiento y análisis
de imágenes médicas.

Realizar una evaluación objetiva basada exclusivamente en la
información visual disponible en la imagen.

No asumir información clínica, antecedentes, síntomas, edad,
sexo ni resultados de otros estudios que no estén disponibles.

Generar la respuesta completamente en español y respetar
EXACTAMENTE la siguiente estructura:


TIPO_ESTUDIO:

Identificar, cuando sea posible, el tipo de estudio observado.

Ejemplos:

- Radiografía de tórax.
- Radiografía abdominal.
- Radiografía musculoesquelética.
- Otro.
- No determinado.


REGION_ANATOMICA:

Indicar la región anatómica principal visible en la imagen.

No inferir regiones que no sean claramente observables.


CALIDAD_TECNICA:

Evaluar brevemente la calidad técnica de la imagen considerando,
cuando sea posi

In [8]:
# ============================================================
# CELDA 17: Inferencia y extraccion con MedGemma via llama.cpp
# ============================================================

import re
import time
from pathlib import Path

from PIL import Image


REQUIRED_REPORT_SECTIONS = (
    "TIPO_ESTUDIO",
    "REGION_ANATOMICA",
    "CALIDAD_TECNICA",
    "HALLAZGOS",
    "HALLAZGO_PRINCIPAL",
    "LOCALIZACION",
    "IMPRESION",
    "CATEGORIA",
    "CONFIANZA",
    "JUSTIFICACION_CATEGORIA",
    "LIMITACIONES",
    "RECOMENDACION",
)


def extract_section(text, section_name):
    """Extrae una seccion del reporte estructurado."""
    headings = (
        "TIPO_ESTUDIO|REGION_ANATOMICA|CALIDAD_TECNICA|HALLAZGOS|"
        "HALLAZGO_PRINCIPAL|LOCALIZACION|IMPRESION|CATEGORIA|CONFIANZA|"
        "JUSTIFICACION_CATEGORIA|LIMITACIONES|RECOMENDACION"
    )
    pattern = (
        rf"(?:^|\n)\s*(?:=+\s*)?(?:#+\s*)?"
        rf"{re.escape(section_name)}\s*:\s*(?:=+\s*)?(.*?)"
    )
    pattern += (
        rf"(?=\n\s*(?:=+\s*)?(?:#+\s*)?"
        rf"(?:{headings})\s*:|\Z)"
    )
    match = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
    if not match:
        return ""
    content = re.sub(r"^\s*=+\s*|\s*=+\s*$", "", match.group(1))
    return content.strip()


def extract_category(text):
    """Extrae una de las tres categorias permitidas."""
    section = extract_section(text, "CATEGORIA")
    match = re.search(
        r"\b(normal|abnormal|indeterminate)\b",
        section,
        flags=re.IGNORECASE,
    )
    return match.group(1).lower() if match else "indeterminate"


def extract_confidence(text):
    """Extrae la confianza declarada por el modelo entre 0 y 100."""
    section = extract_section(text, "CONFIANZA")
    match = re.search(r"\b(100|\d{1,2})\b", section)
    return int(match.group(1)) if match else None


def infer_radiology(image_path):
    """Analiza una imagen con MedGemma (llama.cpp) y devuelve campos."""
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f"No se encontro la imagen: {image_path}")

    start = time.perf_counter()
    try:
        generated_text = llama_chat(
            image_path,
            PROMPT_RADIOLOGIA,
            max_tokens=MAX_NEW_TOKENS,
        )
        latency = time.perf_counter() - start
        peak_vram_gb = llama_vram_gib()
        print(
            f"    Generacion finalizada: {latency:.1f} s "
            f"(VRAM ~{peak_vram_gb} GiB)",
            flush=True,
        )
    except Exception as exc:
        return {
            "filename": image_path.name,
            "path": str(image_path),
            "status": "error",
            "error": str(exc),
            "report": "",
            "category": "indeterminate",
            "confidence": None,
            "latency_seconds": round(time.perf_counter() - start, 3),
            "peak_vram_gb": None,
            "word_count": 0,
            "char_count": 0,
        }

    missing_sections = [
        section for section in REQUIRED_REPORT_SECTIONS
        if not extract_section(generated_text, section)
    ]
    parse_error = not generated_text or bool(missing_sections)

    fields = {
        "tipo_estudio": "TIPO_ESTUDIO",
        "region_anatomica": "REGION_ANATOMICA",
        "calidad_tecnica": "CALIDAD_TECNICA",
        "hallazgos": "HALLAZGOS",
        "hallazgo_principal": "HALLAZGO_PRINCIPAL",
        "localizacion": "LOCALIZACION",
        "impresion": "IMPRESION",
        "justificacion_categoria": "JUSTIFICACION_CATEGORIA",
        "limitaciones": "LIMITACIONES",
        "recomendacion": "RECOMENDACION",
    }

    result = {
        "filename": image_path.name,
        "path": str(image_path),
        "status": "parse_error" if parse_error else "success",
        "error": (
            "Faltan secciones requeridas: " + ", ".join(missing_sections)
            if parse_error else ""
        ),
        "report": generated_text,
        "category": extract_category(generated_text),
        "confidence": extract_confidence(generated_text),
        "latency_seconds": round(latency, 3),
        "peak_vram_gb": round(peak_vram_gb, 3) if peak_vram_gb else None,
        "word_count": len(generated_text.split()),
        "char_count": len(generated_text),
    }
    result.update(
        {key: extract_section(generated_text, heading) for key, heading in fields.items()}
    )
    return result


In [ ]:
# ============================================================
# CELDA 18: Inferencia sobre 10 imágenes del dataset con MedGemma local
# Explicación:
# Selecciona exactamente 10 imágenes del dataset y ejecuta
# MedGemma sobre cada una.
#
# Para cada radiografía se registran:
#
# - Categoría predicha.
# - Nivel de confianza.
# - Tipo de estudio.
# - Región anatómica.
# - Calidad técnica.
# - Hallazgo principal.
# - Localización.
# - Impresión.
# - Latencia de inferencia.
# - Longitud del reporte.
# - Estado del procesamiento.
#
# La selección utiliza una semilla fija para garantizar que
# las mismas 10 imágenes puedan reproducirse posteriormente.
#
# La inferencia se realiza localmente con CUDA y cuantizacion de 4 bits.
# La VRAM maxima se registra para cada muestra.
# ============================================================

import numpy as np
import pandas as pd

from tqdm.notebook import tqdm
from IPython.display import display


# ============================================================
# 1. CONFIGURACIÓN DEL NÚMERO DE IMÁGENES
# ============================================================

NUMERO_IMAGENES_ANALISIS = 10

RANDOM_STATE = 42


# ============================================================
# 2. VALIDACIÓN DEL DATASET
# ============================================================

if "dataset_df" not in globals():

    raise NameError(
        "dataset_df no se encuentra definido. "
        "Debe ejecutarse previamente la celda de carga del dataset."
    )


if dataset_df.empty:

    raise ValueError(
        "dataset_df se encuentra vacío."
    )


if "path" not in dataset_df.columns:

    raise KeyError(
        "La columna 'path' no existe en dataset_df."
    )


if "filename" not in dataset_df.columns:

    raise KeyError(
        "La columna 'filename' no existe en dataset_df."
    )


# ============================================================
# 3. DETERMINACIÓN DEL NÚMERO REAL DE MUESTRAS
# ============================================================

numero_muestras = min(
    NUMERO_IMAGENES_ANALISIS,
    len(dataset_df)
)


# ============================================================
# 4. SELECCIÓN ALEATORIA DE 10 RADIOGRAFÍAS
# ============================================================

sample_df = (
    dataset_df
    .sample(
        n=numero_muestras,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)


print("=" * 80)
print("SELECCIÓN DE IMÁGENES PARA MEDGEMMA")
print("=" * 80)

print(
    "Total de imágenes disponibles:",
    len(dataset_df)
)

print(
    "Imágenes seleccionadas:",
    len(sample_df)
)

print(
    "Semilla de selección:",
    RANDOM_STATE
)

print("=" * 80)


# ============================================================
# 5. VISUALIZACIÓN DE LAS 10 IMÁGENES SELECCIONADAS
# ============================================================

columnas_muestra = [
    column
    for column in [
        "filename",
        "folder",
        "resolution",
        "path"
    ]
    if column in sample_df.columns
]


display(
    sample_df[
        columnas_muestra
    ]
)


# ============================================================
# 6. ESTRUCTURA PARA ALMACENAR RESULTADOS
# ============================================================

evaluation_rows = []


# ============================================================
# 7. PROCESAMIENTO DE LAS 10 IMÁGENES
# ============================================================

for position, row in tqdm(
    sample_df.iterrows(),
    total=len(sample_df),
    desc="Analizando radiografías con MedGemma",
    unit="imagen",
    dynamic_ncols=True
):

    try:

        # ----------------------------------------------------
        # Información de seguimiento
        # ----------------------------------------------------

        print(
            f"\nProcesando imagen "
            f"{position + 1}/{len(sample_df)}: "
            f"{row['filename']}"
        )


        # ----------------------------------------------------
        # Inferencia mediante MedGemma local
        # ----------------------------------------------------

        result = infer_radiology(
            row["path"]
        )

        print(
            f"    Estado: {result.get('status')} | "
            f"Latencia: {result.get('latency_seconds')} s | "
            f"VRAM pico: {result.get('peak_vram_gb')} GB"
        )


        # ----------------------------------------------------
        # Registro del resultado
        # ----------------------------------------------------

        evaluation_rows.append({

            "sample_number":
                position + 1,

            "filename":
                row["filename"],

            "path":
                row["path"],

            "status":
                result.get(
                    "status",
                    "success"
                ),

            "category_pred":
                result.get(
                    "category",
                    "indeterminate"
                ),

            "confidence":
                result.get(
                    "confidence",
                    np.nan
                ),

            "tipo_estudio":
                result.get(
                    "tipo_estudio",
                    ""
                ),

            "region_anatomica":
                result.get(
                    "region_anatomica",
                    ""
                ),

            "calidad_tecnica":
                result.get(
                    "calidad_tecnica",
                    ""
                ),

            "hallazgos":
                result.get(
                    "hallazgos",
                    ""
                ),

            "hallazgo_principal":
                result.get(
                    "hallazgo_principal",
                    ""
                ),

            "localizacion":
                result.get(
                    "localizacion",
                    ""
                ),

            "impresion":
                result.get(
                    "impresion",
                    ""
                ),

            "justificacion_categoria":
                result.get(
                    "justificacion_categoria",
                    ""
                ),

            "limitaciones":
                result.get(
                    "limitaciones",
                    ""
                ),

            "recomendacion":
                result.get(
                    "recomendacion",
                    ""
                ),

            "peak_vram_gb":
                result.get("peak_vram_gb", np.nan),

            "latency_seconds":
                round(
                    result.get(
                        "latency_seconds",
                        np.nan
                    ),
                    3
                ),

            "word_count":
                result.get(
                    "word_count",
                    np.nan
                ),

            "char_count":
                result.get(
                    "char_count",
                    np.nan
                ),

            "report":
                result.get(
                    "report",
                    ""
                ),

            "error":
                result.get(
                    "error",
                    ""
                )

        })


    except Exception as exc:

        # ----------------------------------------------------
        # Registro controlado de errores
        # ----------------------------------------------------

        evaluation_rows.append({

            "sample_number":
                position + 1,

            "filename":
                row["filename"],

            "path":
                row["path"],

            "status":
                "error",

            "category_pred":
                "error",

            "confidence":
                np.nan,

            "tipo_estudio":
                "",

            "region_anatomica":
                "",

            "calidad_tecnica":
                "",

            "hallazgos":
                "",

            "hallazgo_principal":
                "",

            "localizacion":
                "",

            "impresion":
                "",

            "justificacion_categoria":
                "",

            "limitaciones":
                "",

            "recomendacion":
                "",

            "latency_seconds":
                np.nan,

            "peak_vram_gb":
                np.nan,

            "word_count":
                np.nan,

            "char_count":
                np.nan,

            "report":
                "",

            "error":
                str(exc)

        })


# ============================================================
# 8. CREACIÓN DEL DATAFRAME DE RESULTADOS
# ============================================================

evaluation_df = pd.DataFrame(
    evaluation_rows
)

EVALUATION_CSV = REPORTS_DIR / "evaluacion_medgemma.csv"
EVALUATION_JSON = REPORTS_DIR / "evaluacion_medgemma.json"
evaluation_df.to_csv(EVALUATION_CSV, index=False)
evaluation_df.to_json(
    EVALUATION_JSON, orient="records", force_ascii=False, indent=2
)


# ============================================================
# 9. RESUMEN DE PROCESAMIENTO
# ============================================================

total_samples = len(
    evaluation_df
)


successful_samples = (
    evaluation_df[
        "status"
    ]
    .eq("success")
    .sum()
)


error_samples = (
    evaluation_df[
        "status"
    ]
    .ne("success")
    .sum()
)


print()

print("=" * 80)

print(
    "INFERENCIA MEDIANTE MEDGEMMA FINALIZADA"
)

print("=" * 80)

print(
    "Modelo:",
    MODEL_ID
)

print(
    "Imágenes programadas:",
    numero_muestras
)

print(
    "Imágenes procesadas:",
    total_samples
)

print(
    "Procesamientos exitosos:",
    successful_samples
)

print(
    "Procesamientos con error:",
    error_samples
)


# ============================================================
# 10. MÉTRICAS OPERATIVAS PRELIMINARES
# ============================================================

if successful_samples > 0:

    successful_df = evaluation_df[
        evaluation_df[
            "status"
        ] == "success"
    ].copy()


    print()

    print(
        "Latencia promedio:",
        round(
            successful_df[
                "latency_seconds"
            ].mean(),
            3
        ),
        "segundos"
    )


    print(
        "Latencia mínima:",
        round(
            successful_df[
                "latency_seconds"
            ].min(),
            3
        ),
        "segundos"
    )


    print(
        "Latencia máxima:",
        round(
            successful_df[
                "latency_seconds"
            ].max(),
            3
        ),
        "segundos"
    )


    print(
        "Latencia mediana:",
        round(
            successful_df[
                "latency_seconds"
            ].median(),
            3
        ),
        "segundos"
    )


    print(
        "Palabras promedio por reporte:",
        round(
            successful_df[
                "word_count"
            ].mean(),
            2
        )
    )


    if successful_df[
        "confidence"
    ].notna().any():

        print(
            "Confianza promedio:",
            round(
                successful_df[
                    "confidence"
                ].mean(),
                2
            )
        )


# ============================================================
# 11. DISTRIBUCIÓN DE CATEGORÍAS
# ============================================================

if successful_samples > 0:

    print()

    print("=" * 80)

    print(
        "DISTRIBUCIÓN DE CATEGORÍAS"
    )

    print("=" * 80)


    category_distribution = (
        successful_df[
            "category_pred"
        ]
        .value_counts(
            dropna=False
        )
        .rename_axis(
            "categoria"
        )
        .reset_index(
            name="cantidad"
        )
    )


    display(
        category_distribution
    )


# ============================================================
# 12. VISUALIZACIÓN DE RESULTADOS
# ============================================================

columns_to_display = [

    "sample_number",

    "filename",

    "status",

    "category_pred",

    "confidence",

    "tipo_estudio",

    "region_anatomica",

    "hallazgos",

    "hallazgo_principal",

    "localizacion",

    "latency_seconds",

    "peak_vram_gb",

    "word_count",

    "char_count",

    "error"

]


print()

print("=" * 80)

print(
    "RESULTADOS DE LAS 10 RADIOGRAFÍAS"
)

print("=" * 80)


display(

    evaluation_df[
        columns_to_display
    ]

)

In [ ]:
# ============================================================
# CELDA NUEVA: Re-extracción de campos desde los reportes
# Explicación:
# Repuebla los campos estructurados (categoría, confianza y
# secciones) directamente desde la columna "report" ya generada
# por MedGemma, sin volver a ejecutar el modelo.
#
# Requisito: ejecutar primero la CELDA 17 (funciones de
# extracción corregidas) y tener evaluation_df generado.
# ============================================================

if "evaluation_df" not in globals():

    raise NameError(
        "evaluation_df no está definido. "
        "Ejecuta primero la celda de inferencia."
    )


for idx in evaluation_df.index:

    if evaluation_df.at[idx, "status"] == "error":

        continue


    report_text = evaluation_df.at[idx, "report"]


    missing_sections = [
        section for section in REQUIRED_REPORT_SECTIONS
        if not extract_section(report_text, section)
    ]
    evaluation_df.at[idx, "status"] = (
        "parse_error" if missing_sections else "success"
    )
    evaluation_df.at[idx, "error"] = (
        "Faltan secciones requeridas: " + ", ".join(missing_sections)
        if missing_sections else ""
    )


    evaluation_df.at[idx, "category_pred"] = extract_category(report_text)

    evaluation_df.at[idx, "confidence"] = extract_confidence(report_text)

    evaluation_df.at[idx, "tipo_estudio"] = extract_section(report_text, "TIPO_ESTUDIO")

    evaluation_df.at[idx, "region_anatomica"] = extract_section(report_text, "REGION_ANATOMICA")

    evaluation_df.at[idx, "calidad_tecnica"] = extract_section(report_text, "CALIDAD_TECNICA")

    evaluation_df.at[idx, "hallazgos"] = extract_section(report_text, "HALLAZGOS")

    evaluation_df.at[idx, "hallazgo_principal"] = extract_section(report_text, "HALLAZGO_PRINCIPAL")

    evaluation_df.at[idx, "localizacion"] = extract_section(report_text, "LOCALIZACION")

    evaluation_df.at[idx, "impresion"] = extract_section(report_text, "IMPRESION")

    evaluation_df.at[idx, "justificacion_categoria"] = extract_section(report_text, "JUSTIFICACION_CATEGORIA")

    evaluation_df.at[idx, "limitaciones"] = extract_section(report_text, "LIMITACIONES")

    evaluation_df.at[idx, "recomendacion"] = extract_section(report_text, "RECOMENDACION")


evaluation_df.to_csv(EVALUATION_CSV, index=False)
evaluation_df.to_json(
    EVALUATION_JSON, orient="records", force_ascii=False, indent=2
)


display(
    evaluation_df[
        [
            "filename",
            "category_pred",
            "confidence",
            "tipo_estudio",
            "hallazgo_principal",
            "impresion"
        ]
    ]
)


In [ ]:
# ============================================================
# CELDA 19: Métricas operativas de inferencia con MedGemma local
# Explicación:
# Resume el comportamiento operativo del análisis realizado
# mediante MedGemma local sobre las imágenes seleccionadas.
#
# Las métricas incluyen:
#
# - Número total de imágenes evaluadas.
# - Número de inferencias ejecutadas correctamente.
# - Número de errores.
# - Tasa de ejecución correcta.
# - Latencia media.
# - Latencia mediana.
# - Latencia mínima.
# - Latencia máxima.
# - Desviación estándar de la latencia.
# - Palabras medias por reporte.
# - Caracteres medios por reporte.
# - Confianza media declarada por MedGemma.
#
# La inferencia se ejecuta localmente sobre la GPU disponible.
# ============================================================

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 1. VALIDACIÓN DEL DATAFRAME DE EVALUACIÓN
# ============================================================

if "evaluation_df" not in globals():

    raise NameError(
        "evaluation_df no se encuentra definido. "
        "Debe ejecutarse previamente la celda de inferencia."
    )


if evaluation_df.empty:

    raise ValueError(
        "evaluation_df se encuentra vacío."
    )


# ============================================================
# 2. IDENTIFICACIÓN DE EJECUCIONES CORRECTAS
# ============================================================

if "status" in evaluation_df.columns:

    successful_df = evaluation_df[
        evaluation_df["status"] == "success"
    ].copy()

else:

    successful_df = evaluation_df[
        evaluation_df["error"].fillna("") == ""
    ].copy()


# ============================================================
# 3. IDENTIFICACIÓN DE ERRORES
# ============================================================

error_df = evaluation_df[
    ~evaluation_df.index.isin(
        successful_df.index
    )
].copy()


# ============================================================
# 4. VARIABLES GENERALES
# ============================================================

total_samples = len(
    evaluation_df
)

successful_samples = len(
    successful_df
)

error_samples = len(
    error_df
)


# ============================================================
# 5. TASA DE EJECUCIÓN CORRECTA
# ============================================================

success_rate = (

    successful_samples
    / total_samples

    if total_samples > 0

    else np.nan
)


# ============================================================
# 6. MÉTRICAS DE LATENCIA
# ============================================================

if successful_samples > 0:

    latency_mean = (
        successful_df[
            "latency_seconds"
        ].mean()
    )

    latency_median = (
        successful_df[
            "latency_seconds"
        ].median()
    )

    latency_min = (
        successful_df[
            "latency_seconds"
        ].min()
    )

    latency_max = (
        successful_df[
            "latency_seconds"
        ].max()
    )

    latency_std = (
        successful_df[
            "latency_seconds"
        ].std()
    )

else:

    latency_mean = np.nan
    latency_median = np.nan
    latency_min = np.nan
    latency_max = np.nan
    latency_std = np.nan


# ============================================================
# 7. MÉTRICAS DE LONGITUD DE LOS REPORTES
# ============================================================

if successful_samples > 0:

    mean_words = (
        successful_df[
            "word_count"
        ].mean()
    )

    mean_chars = (
        successful_df[
            "char_count"
        ].mean()
    )

else:

    mean_words = np.nan
    mean_chars = np.nan


# ============================================================
# 8. MÉTRICA DE CONFIANZA
# ============================================================

if (
    successful_samples > 0
    and "confidence" in successful_df.columns
    and successful_df["confidence"].notna().any()
):

    mean_confidence = (
        successful_df[
            "confidence"
        ].mean()
    )

    median_confidence = (
        successful_df[
            "confidence"
        ].median()
    )

else:

    mean_confidence = np.nan
    median_confidence = np.nan


# ============================================================
# 9. CONSTRUCCIÓN DE LA TABLA DE MÉTRICAS
# ============================================================

operational_metrics = pd.DataFrame({

    "Métrica": [

        "Imágenes evaluadas",

        "Inferencias correctas",

        "Inferencias con error",

        "Tasa de ejecución correcta",

        "Latencia media (s)",

        "Latencia mediana (s)",

        "Latencia mínima (s)",

        "Latencia máxima (s)",

        "Desviación estándar de latencia (s)",

        "Palabras medias por reporte",

        "Caracteres medios por reporte",

        "Confianza media",

        "Confianza mediana"

    ],

    "Valor": [

        total_samples,

        successful_samples,

        error_samples,

        round(
            success_rate,
            4
        )
        if not np.isnan(success_rate)
        else np.nan,

        round(
            latency_mean,
            3
        )
        if not np.isnan(latency_mean)
        else np.nan,

        round(
            latency_median,
            3
        )
        if not np.isnan(latency_median)
        else np.nan,

        round(
            latency_min,
            3
        )
        if not np.isnan(latency_min)
        else np.nan,

        round(
            latency_max,
            3
        )
        if not np.isnan(latency_max)
        else np.nan,

        round(
            latency_std,
            3
        )
        if not np.isnan(latency_std)
        else np.nan,

        round(
            mean_words,
            2
        )
        if not np.isnan(mean_words)
        else np.nan,

        round(
            mean_chars,
            2
        )
        if not np.isnan(mean_chars)
        else np.nan,

        round(
            mean_confidence,
            2
        )
        if not np.isnan(mean_confidence)
        else np.nan,

        round(
            median_confidence,
            2
        )
        if not np.isnan(median_confidence)
        else np.nan

    ]

})


# ============================================================
# 10. PRESENTACIÓN DE RESULTADOS
# ============================================================

print("=" * 80)

print(
    "MÉTRICAS OPERATIVAS DE MEDGEMMA LOCAL"
)

print("=" * 80)

print()

print(
    "Modelo:",
    MODEL_ID
)

print(
    "Imágenes evaluadas:",
    total_samples
)

print(
    "Inferencias correctas:",
    successful_samples
)

print(
    "Inferencias con error:",
    error_samples
)

print()

display(
    operational_metrics
)

In [ ]:
# ============================================================
# CELDA 26: Análisis individual de las 10 imágenes
# Explicación:
# Presenta de forma estructurada el resultado obtenido para
# cada una de las 10 radiografías analizadas con MedGemma local.
#
# Para cada imagen se muestran:
#
# - Número de muestra.
# - Nombre del archivo.
# - Estado de ejecución.
# - Categoría predicha.
# - Nivel de confianza.
# - Tipo de estudio.
# - Región anatómica.
# - Calidad técnica.
# - Hallazgo principal.
# - Localización.
# - Impresión.
# - Justificación de la categoría.
# - Limitaciones.
# - Recomendación.
# - Latencia de inferencia.
#
# El objetivo es facilitar la interpretación académica
# individual antes de realizar el análisis global.
# ============================================================

import pandas as pd
import numpy as np

from IPython.display import display, Markdown


# ============================================================
# 1. VALIDACIÓN DE LOS RESULTADOS
# ============================================================

if "evaluation_df" not in globals():

    raise NameError(
        "evaluation_df no se encuentra definido. "
        "Debe ejecutarse previamente la celda de inferencia."
    )


if evaluation_df.empty:

    raise ValueError(
        "evaluation_df no contiene resultados."
    )


# ============================================================
# 2. SELECCIÓN DE LAS PRIMERAS 10 IMÁGENES
# ============================================================

individual_analysis_df = (
    evaluation_df
    .head(10)
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 3. FUNCIÓN DE LIMPIEZA DE TEXTO
# ============================================================

def clean_value(value):
    """
    Normaliza valores vacíos o nulos para mejorar
    la presentación de los resultados.
    """

    if pd.isna(value):

        return "No disponible"


    value = str(value).strip()


    if value == "":

        return "No disponible"


    return value


# ============================================================
# 4. FUNCIÓN PARA INTERPRETAR EL NIVEL DE CONFIANZA
# ============================================================

def interpret_confidence(value):
    """
    Clasifica descriptivamente el nivel de confianza
    declarado por MedGemma.
    """

    try:

        value = float(value)

    except (TypeError, ValueError):

        return "No disponible"


    if value < 40:

        return "Baja"

    elif value < 70:

        return "Moderada"

    elif value < 90:

        return "Alta"

    else:

        return "Muy alta"


# ============================================================
# 5. RECORRIDO INDIVIDUAL DE LAS 10 IMÁGENES
# ============================================================

for index, row in individual_analysis_df.iterrows():

    sample_number = (
        row.get(
            "sample_number",
            index + 1
        )
    )


    filename = clean_value(
        row.get(
            "filename",
            ""
        )
    )


    status = clean_value(
        row.get(
            "status",
            ""
        )
    )


    category = clean_value(
        row.get(
            "category_pred",
            ""
        )
    )


    confidence = row.get(
        "confidence",
        np.nan
    )


    confidence_label = interpret_confidence(
        confidence
    )


    tipo_estudio = clean_value(
        row.get(
            "tipo_estudio",
            ""
        )
    )


    region_anatomica = clean_value(
        row.get(
            "region_anatomica",
            ""
        )
    )


    calidad_tecnica = clean_value(
        row.get(
            "calidad_tecnica",
            ""
        )
    )


    hallazgo_principal = clean_value(
        row.get(
            "hallazgo_principal",
            ""
        )
    )


    localizacion = clean_value(
        row.get(
            "localizacion",
            ""
        )
    )


    impresion = clean_value(
        row.get(
            "impresion",
            ""
        )
    )


    justificacion = clean_value(
        row.get(
            "justificacion_categoria",
            ""
        )
    )


    limitaciones = clean_value(
        row.get(
            "limitaciones",
            ""
        )
    )


    recomendacion = clean_value(
        row.get(
            "recomendacion",
            ""
        )
    )


    latency = row.get(
        "latency_seconds",
        np.nan
    )


    error = clean_value(
        row.get(
            "error",
            ""
        )
    )


    # ========================================================
    # 6. PRESENTACIÓN DEL RESULTADO INDIVIDUAL
    # ========================================================

    display(
        Markdown(
            f"""
## Imagen {sample_number} de {len(individual_analysis_df)}

**Archivo:** `{filename}`

**Estado de ejecución:** {status}

**Categoría predicha:** `{category}`

**Nivel de confianza:** {confidence if pd.notna(confidence) else "No disponible"} %

**Interpretación de confianza:** {confidence_label}

### Tipo de estudio
{tipo_estudio}

### Región anatómica
{region_anatomica}

### Calidad técnica
{calidad_tecnica}

### Hallazgo principal
{hallazgo_principal}

### Localización
{localizacion}

### Impresión radiológica
{impresion}

### Justificación de la categoría
{justificacion}

### Limitaciones
{limitaciones}

### Recomendación
{recomendacion}

### Tiempo de inferencia
{round(latency, 3) if pd.notna(latency) else "No disponible"} segundos

### Error
{error}

---
"""
        )
    )

In [ ]:
# ============================================================
# CELDA 28: Síntesis de voz del reporte radiológico con Gemini
# Explicación:
# Convierte el reporte radiológico de una de las imágenes
# analizadas correctamente en un archivo de audio WAV.
#
# El proceso utiliza Gemini TTS mediante API.
#
# El audio recibido corresponde a datos PCM, por lo que se
# construye un archivo WAV con:
#
# - 1 canal (mono).
# - 24 000 Hz.
# - 16 bits por muestra.
#
# También se registra:
#
# - Imagen asociada.
# - Modelo TTS utilizado.
# - Duración del procesamiento.
# - Tamaño del archivo generado.
#
# La síntesis de voz constituye una representación auditiva
# del reporte automatizado y no sustituye una interpretación
# radiológica profesional.
# ============================================================


# ============================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import time
import wave

from pathlib import Path
from IPython.display import Audio, display

from google import genai
from google.genai import types


# ============================================================
# 2. CONFIGURACIÓN DEL MODELO TTS
# ============================================================

GEMINI_TTS_MODEL = os.getenv(
    "GEMINI_TTS_MODEL",
    "gemini-2.5-flash-preview-tts",
)

TTS_VOICE = "Kore"

TTS_SAMPLE_RATE = 24000

TTS_CHANNELS = 1

TTS_SAMPLE_WIDTH = 2


# ============================================================
# 3. VALIDACIÓN DE LA CLAVE API
# ============================================================

if not GEMINI_API_KEY:

    print(
        "TTS omitida: GEMINI_API_KEY no se encuentra configurada."
    )


# ============================================================
# 4. CREACIÓN DEL CLIENTE GEMINI
# ============================================================

tts_client = (
    genai.Client(api_key=GEMINI_API_KEY)
    if GEMINI_API_KEY
    else None
)


# ============================================================
# 5. VALIDACIÓN DE LOS RESULTADOS DE INFERENCIA
# ============================================================

if "evaluation_df" not in globals():

    raise NameError(
        "evaluation_df no se encuentra definido. "
        "Debe ejecutarse previamente la inferencia "
        "de las imágenes."
    )


if evaluation_df.empty:

    raise ValueError(
        "evaluation_df no contiene resultados."
    )


# ============================================================
# 6. SELECCIÓN DE REPORTES EXITOSOS
# ============================================================

if "status" in evaluation_df.columns:

    tts_successful_df = evaluation_df[
        evaluation_df["status"] == "success"
    ].copy()

else:

    tts_successful_df = evaluation_df[
        evaluation_df["error"].fillna("") == ""
    ].copy()


# ============================================================
# 7. VALIDACIÓN DE REPORTES
# ============================================================

tts_successful_df = tts_successful_df[
    tts_successful_df["report"]
    .fillna("")
    .str.strip()
    .ne("")
].copy()


if tts_successful_df.empty:

    print(
        "TTS omitida: no existen reportes validos para generar audio."
    )
    tts_client = None


# ============================================================
# 8. SELECCIÓN DEL PRIMER REPORTE EXITOSO
# ============================================================

selected_report = (
    tts_successful_df.iloc[0]
    if not tts_successful_df.empty
    else {"filename": "sin_reporte", "report": ""}
)

filename = selected_report["filename"]

category = selected_report.get(
    "category_pred",
    "No disponible"
)

confidence = selected_report.get(
    "confidence",
    "No disponible"
)

report_text = str(
    selected_report["report"]
).strip()


# ============================================================
# 9. LIMITACIÓN DEL TEXTO PARA TTS
# ============================================================

MAX_TTS_CHARACTERS = 5000

report_truncated = len(report_text) > MAX_TTS_CHARACTERS

if report_truncated:

    shortened_report = report_text[:MAX_TTS_CHARACTERS]
    sentence_end = max(
        shortened_report.rfind("."),
        shortened_report.rfind("\n"),
    )
    report_text = (
        shortened_report[:sentence_end + 1]
        if sentence_end >= MAX_TTS_CHARACTERS // 2
        else shortened_report
    ).strip()

truncation_notice = (
    "El reporte fue abreviado para la sintesis de voz."
    if report_truncated
    else ""
)


# ============================================================
# 10. CONSTRUCCIÓN DEL TEXTO PARA AUDIO
# ============================================================

tts_text = f"""
Reporte automatizado de análisis radiológico.

Archivo analizado:
{filename}.

Categoría generada:
{category}.

Nivel de confianza:
{confidence} por ciento.

Reporte radiológico:

{report_text}

{truncation_notice}

Fin del reporte.

Este resultado corresponde a un análisis automatizado
experimental y no sustituye la interpretación realizada
por un profesional cualificado.
""".strip()


# ============================================================
# 11. DEFINICIÓN DE LA RUTA DE SALIDA
# ============================================================

AUDIO_DIR = (
    OUTPUT_DIR
    / "audio"
)

AUDIO_DIR.mkdir(
    parents=True,
    exist_ok=True
)


audio_path = (
    AUDIO_DIR
    / f"{Path(filename).stem}_reporte_gemini.wav"
)


# ============================================================
# 12. FUNCIÓN PARA GUARDAR PCM COMO WAV
# ============================================================

def save_pcm_as_wav(
    filename,
    pcm_data,
    channels=1,
    sample_rate=24000,
    sample_width=2
):

    with wave.open(
        str(filename),
        "wb"
    ) as wav_file:

        wav_file.setnchannels(
            channels
        )

        wav_file.setsampwidth(
            sample_width
        )

        wav_file.setframerate(
            sample_rate
        )

        wav_file.writeframes(
            pcm_data
        )


# ============================================================
# 13. GENERACIÓN DEL AUDIO CON GEMINI TTS
# ============================================================

print("=" * 80)

print(
    "GENERACIÓN DEL REPORTE EN AUDIO"
)

print("=" * 80)

print()

print(
    "Imagen:",
    filename
)

print(
    "Categoría:",
    category
)

print(
    "Confianza:",
    confidence
)

print(
    "Modelo TTS:",
    GEMINI_TTS_MODEL
)

print(
    "Voz:",
    TTS_VOICE
)

print()

print(
    "Generando audio..."
)


tts_start = time.perf_counter()


try:

    if tts_client is None:

        raise RuntimeError(
            "TTS omitida porque GEMINI_API_KEY no esta configurada."
        )

    response = tts_client.models.generate_content(

        model=GEMINI_TTS_MODEL,

        contents=tts_text,

        config=types.GenerateContentConfig(

            response_modalities=[
                "AUDIO"
            ],

            speech_config=types.SpeechConfig(

                voice_config=types.VoiceConfig(

                    prebuilt_voice_config=
                    types.PrebuiltVoiceConfig(

                        voice_name=TTS_VOICE

                    )

                )

            )

        )

    )


    # ========================================================
    # 14. EXTRACCIÓN DE LOS DATOS PCM
    # ========================================================

    audio_bytes = (

        response
        .candidates[0]
        .content
        .parts[0]
        .inline_data
        .data

    )


    if not audio_bytes:

        raise ValueError(
            "Gemini no devolvió datos de audio."
        )


    # ========================================================
    # 15. CONVERSIÓN DE PCM A WAV
    # ========================================================

    save_pcm_as_wav(

        filename=audio_path,

        pcm_data=audio_bytes,

        channels=TTS_CHANNELS,

        sample_rate=TTS_SAMPLE_RATE,

        sample_width=TTS_SAMPLE_WIDTH

    )


    # ========================================================
    # 16. TIEMPO TOTAL
    # ========================================================

    tts_seconds = (

        time.perf_counter()
        - tts_start

    )


    # ========================================================
    # 17. TAMAÑO DEL ARCHIVO
    # ========================================================

    audio_size_mb = (

        audio_path.stat().st_size
        / (1024 ** 2)

    )


    # ========================================================
    # 18. PRESENTACIÓN DE RESULTADOS
    # ========================================================

    print()

    print("=" * 80)

    print(
        "SÍNTESIS DE VOZ FINALIZADA"
    )

    print("=" * 80)

    print()

    print(
        "Archivo analizado:",
        filename
    )

    print(
        "Modelo:",
        GEMINI_TTS_MODEL
    )

    print(
        "Voz:",
        TTS_VOICE
    )

    print(
        "Frecuencia:",
        f"{TTS_SAMPLE_RATE} Hz"
    )

    print(
        "Canales:",
        TTS_CHANNELS
    )

    print(
        "Profundidad:",
        f"{TTS_SAMPLE_WIDTH * 8} bits"
    )

    print(
        "Tiempo TTS:",
        round(
            tts_seconds,
            2
        ),
        "segundos"
    )

    print(
        "Tamaño del audio:",
        round(
            audio_size_mb,
            3
        ),
        "MB"
    )

    print()

    print(
        "Audio almacenado en:"
    )

    print(
        audio_path
    )


    # ========================================================
    # 19. REPRODUCTOR DE AUDIO EN JUPYTER
    # ========================================================

    print()

    print(
        "Reproducción del reporte:"
    )

    display(

        Audio(
            filename=str(
                audio_path
            )
        )

    )


except Exception as exc:

    tts_seconds = (

        time.perf_counter()
        - tts_start

    )


    print()

    print("=" * 80)

    print(
        "ERROR EN LA SÍNTESIS DE VOZ"
    )

    print("=" * 80)

    print()

    print(
        "Tipo de error:",
        type(exc).__name__
    )

    print(
        "Descripción:",
        str(exc)
    )

    print(
        "Tiempo transcurrido:",
        round(
            tts_seconds,
            2
        ),
        "segundos"
    )

In [ ]:
# ============================================================
# CORRECCIÓN CELDA 29:
# COMPONENTE PARA CARGAR IMAGEN DESDE EL COMPUTADOR
# ============================================================

uploaded_image = gr.Image(

    # La función recibirá directamente la ruta temporal
    # creada por Gradio.
    type="filepath",

    # Solamente carga desde archivos.
    sources=["upload"],

    label="Cargar imagen radiológica",

    height=450,

    interactive=True

)

In [ ]:
# ============================================================
# FUNCIÓN PRINCIPAL CORREGIDA
# ============================================================

def process_gradio(
    source,
    uploaded_image,
    dataset_index
):

    try:

        # ====================================================
        # OPCIÓN 1: CARGAR IMAGEN DESDE EL COMPUTADOR
        # ====================================================

        if source == "Cargar imagen":

            if uploaded_image is None:

                raise ValueError(
                    "Debe seleccionarse una imagen radiológica."
                )


            # ------------------------------------------------
            # Gradio entrega directamente una ruta temporal.
            # ------------------------------------------------

            analysis_path = Path(
                uploaded_image
            )


            if not analysis_path.exists():

                raise FileNotFoundError(
                    f"No fue posible localizar la imagen "
                    f"cargada:\n{analysis_path}"
                )


            source_name = (
                "Imagen cargada manualmente"
            )


            original_filename = (
                analysis_path.name
            )


            # ------------------------------------------------
            # Vista previa
            # ------------------------------------------------

            with Image.open(
                analysis_path
            ) as img:

                preview_image = (
                    img.convert("RGB").copy()
                )


        # ====================================================
        # OPCIÓN 2: SELECCIONAR IMAGEN DEL DATASET
        # ====================================================

        elif source == "Seleccionar del dataset":

            if dataset_index in [
                None,
                ""
            ]:

                raise ValueError(
                    "Debe seleccionarse una imagen "
                    "del dataset."
                )


            index = int(
                dataset_index
            )


            row = dataset_df.iloc[
                index
            ]


            analysis_path = Path(
                row["path"]
            )


            if not analysis_path.exists():

                raise FileNotFoundError(
                    f"No se encontró la imagen:\n"
                    f"{analysis_path}"
                )


            source_name = (
                "Dataset"
            )


            original_filename = str(
                row["filename"]
            )


            with Image.open(
                analysis_path
            ) as img:

                preview_image = (
                    img.convert("RGB").copy()
                )


        else:

            raise ValueError(
                "Debe seleccionarse una fuente "
                "de imagen válida."
            )


        # ====================================================
        # INFORMACIÓN EN CONSOLA
        # ====================================================

        print("=" * 80)

        print(
            "IMAGEN RECIBIDA"
        )

        print("=" * 80)

        print(
            "Fuente:",
            source_name
        )

        print(
            "Archivo:",
            original_filename
        )

        print(
            "Ruta:",
            analysis_path
        )

        print(
            "Existe:",
            analysis_path.exists()
        )

        print()


        # ====================================================
        # INFERENCIA
        # ====================================================

        result = infer_radiology(
            analysis_path
        )


        # ====================================================
        # CONTROL DE ERROR DE MEDGEMMA
        # ====================================================

        if result.get(
            "status"
        ) != "success":

            raise RuntimeError(
                result.get(
                    "error",
                    "Error durante la inferencia."
                )
            )


        # ====================================================
        # MÉTRICAS
        # ====================================================

        metrics = pd.DataFrame({
            "Métrica": [
                "Modelo",
                "Fuente",
                "Archivo",
                "Categoría",
                "Confianza (%)",
                "Latencia (s)",
                "VRAM máxima (GB)",
                "Palabras",
                "Caracteres",
                "Estado",
            ],
            "Valor": [
                MODEL_ID,
                source_name,
                original_filename,
                result.get("category", "indeterminate"),
                result.get("confidence", np.nan),
                round(result.get("latency_seconds", np.nan), 3),
                result.get("peak_vram_gb", np.nan),
                result.get("word_count", np.nan),
                result.get("char_count", np.nan),
                result.get("status", "error"),
            ],
        })


        # ====================================================
        # GUARDADO
        # ====================================================

        (
            saved_image,
            saved_report,
            results_df

        ) = save_gradio_result(

            result=result,

            image_path=analysis_path,

            source=source_name,

            original_filename=original_filename

        )


        save_message = (

            "Resultado almacenado correctamente.\n\n"

            f"Imagen:\n{saved_image}\n\n"

            f"Reporte:\n{saved_report}\n\n"

            f"CSV:\n{GRADIO_RESULTS_FILE}\n\n"

            f"Total de análisis almacenados: "
            f"{len(results_df)}"

        )


        # ====================================================
        # SALIDAS DE GRADIO
        # ====================================================

        return (

            preview_image,

            result.get(
                "category",
                "indeterminate"
            ),

            result.get(
                "confidence",
                "No disponible"
            ),

            result.get(
                "tipo_estudio",
                ""
            ),

            result.get(
                "region_anatomica",
                ""
            ),

            result.get(
                "calidad_tecnica",
                ""
            ),

            result.get(
                "hallazgo_principal",
                ""
            ),

            result.get(
                "localizacion",
                ""
            ),

            result.get(
                "impresion",
                ""
            ),

            result.get(
                "report",
                ""
            ),

            metrics,

            save_message

        )


    except Exception as exc:

        error_metrics = pd.DataFrame({

            "Métrica": [
                "Estado",
                "Error"
            ],

            "Valor": [
                "error",
                str(exc)
            ]

        })


        return (

            None,

            "error",

            "No disponible",

            "No disponible",

            "No disponible",

            "No disponible",

            "No disponible",

            "No disponible",

            "No disponible",

            f"Error durante el análisis:\n{exc}",

            error_metrics,

            "No se almacenaron resultados."

        )

In [ ]:
# ============================================================
# CELDA 29: Construcción de la interfaz Gradio
# Explicación:
# Define los directorios de salida, la función de guardado de
# resultados y la interfaz Gradio que une process_gradio con
# los componentes de entrada y salida.
# ============================================================

from datetime import datetime
from pathlib import Path
from threading import Lock

from PIL import Image

import pandas as pd

import gradio as gr


# ============================================================
# 1. DIRECTORIOS Y ARCHIVOS DE RESULTADOS
# ============================================================

GRADIO_OUTPUT_DIR = OUTPUT_DIR / "gradio"

GRADIO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


GRADIO_RESULTS_FILE = GRADIO_OUTPUT_DIR / "resultados_gradio.csv"
GRADIO_SAVE_LOCK = Lock()


# ============================================================
# 2. FUNCIÓN DE GUARDADO DE RESULTADOS
# ============================================================

def save_gradio_result(
    result,
    image_path,
    source,
    original_filename
):

    stem = Path(original_filename).stem

    run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

    saved_image = GRADIO_OUTPUT_DIR / f"{stem}_{run_id}_imagen.png"

    saved_report = GRADIO_OUTPUT_DIR / f"{stem}_{run_id}_reporte.txt"


    with Image.open(
        image_path
    ) as img:

        img.convert("RGB").save(
            saved_image
        )


    saved_report.write_text(
        str(result.get("report", "")),
        encoding="utf-8"
    )


    row = {

        "fecha":
            datetime.now().isoformat(timespec="seconds"),

        "fuente":
            source,

        "archivo":
            original_filename,

        "categoria":
            result.get("category", "indeterminate"),

        "confianza":
            result.get("confidence"),

        "tipo_estudio":
            result.get("tipo_estudio", ""),

        "region_anatomica":
            result.get("region_anatomica", ""),

        "hallazgos":
            result.get("hallazgos", ""),

        "impresion":
            result.get("impresion", ""),

        "latencia_s":
            result.get("latency_seconds"),

        "vram_maxima_gb":
            result.get("peak_vram_gb"),

        "palabras":
            result.get("word_count"),

        "imagen_guardada":
            str(saved_image),

        "reporte_guardado":
            str(saved_report)

    }


    with GRADIO_SAVE_LOCK:

        if GRADIO_RESULTS_FILE.exists():

            results_df = pd.read_csv(
                GRADIO_RESULTS_FILE
            )

            results_df = pd.concat(
                [
                    results_df,
                    pd.DataFrame([row])
                ],
                ignore_index=True
            )

        else:

            results_df = pd.DataFrame([row])


        results_df.to_csv(
            GRADIO_RESULTS_FILE,
            index=False
        )


    return (
        str(saved_image),
        str(saved_report),
        results_df
    )


# ============================================================
# 3. COMPONENTES DE ENTRADA
# ============================================================

source_component = gr.Radio(
    choices=[
        "Cargar imagen",
        "Seleccionar del dataset"
    ],
    value="Cargar imagen",
    label="Fuente de la imagen"
)


if "dataset_df" in globals():

    dataset_choices = [
        (dataset_df.iloc[i]["filename"], i)
        for i in range(len(dataset_df))
    ]

else:

    dataset_choices = []


dataset_index_component = gr.Dropdown(
    choices=dataset_choices,
    label="Imagen del dataset (índice)"
)


# ============================================================
# 4. COMPONENTES DE SALIDA
# ============================================================

outputs = [

    gr.Image(label="Vista previa"),

    gr.Textbox(label="Categoría"),

    gr.Textbox(label="Confianza"),

    gr.Textbox(label="Tipo de estudio"),

    gr.Textbox(label="Región anatómica"),

    gr.Textbox(label="Calidad técnica"),

    gr.Textbox(label="Hallazgo principal"),

    gr.Textbox(label="Localización"),

    gr.Textbox(label="Impresión"),

    gr.Textbox(label="Reporte completo"),

    gr.Dataframe(label="Métricas"),

    gr.Textbox(label="Guardado")

]


# ============================================================
# 5. CONSTRUCCIÓN DE LA INTERFAZ
# ============================================================

interface = gr.Interface(

    fn=process_gradio,

    inputs=[
        source_component,
        uploaded_image,
        dataset_index_component
    ],

    outputs=outputs,

    title="Análisis radiológico con MedGemma",

    description=(
        "Carga una imagen radiológica o selecciona una del "
        "dataset para obtener un análisis automatizado "
        "experimental mediante MedGemma local."
    )

)


In [ ]:
# ============================================================
# CELDA 30: Lanzamiento de la interfaz Gradio
# Explicación:
# Inicia la aplicación gráfica de análisis radiológico
# desarrollada en la CELDA 29.
#
# La interfaz permite:
#
# 1. Cargar una imagen radiológica desde el computador.
# 2. Seleccionar una imagen directamente desde el dataset.
# 3. Visualizar la imagen antes del procesamiento.
# 4. Ejecutar el análisis mediante MedGemma local.
# 5. Mostrar categoría, confianza, hallazgos y reporte.
# 6. Guardar automáticamente los resultados.
#
# La aplicación se ejecuta localmente mediante Gradio.
# ============================================================


# ============================================================
# 1. VALIDACIÓN DE LA INTERFAZ
# ============================================================

if "interface" not in globals():

    raise NameError(
        "La interfaz Gradio no se encuentra definida. "
        "Debe ejecutarse previamente la CELDA 29."
    )


# ============================================================
# 2. VALIDACIÓN DEL COMPONENTE PARA CARGAR IMÁGENES
# ============================================================

if "uploaded_image" not in globals():

    raise NameError(
        "El componente para cargar imágenes no se encuentra "
        "definido en la interfaz. Debe existir en la CELDA 29 "
        "un componente gr.Image(type='pil')."
    )


# ============================================================
# 3. VALIDACIÓN DEL DIRECTORIO DE SALIDA
# ============================================================

if "OUTPUT_DIR" not in globals():

    raise NameError(
        "OUTPUT_DIR no se encuentra definido."
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 4. VALIDACIÓN DEL DIRECTORIO DEL DATASET
# ============================================================

allowed_directories = [
    str(
        OUTPUT_DIR.resolve()
    )
]


if "DATASET_DIR" in globals():

    if DATASET_DIR.exists():

        allowed_directories.append(
            str(
                DATASET_DIR.resolve()
            )
        )


# ============================================================
# 5. INFORMACIÓN DEL SISTEMA
# ============================================================

print("=" * 80)

print(
    "SISTEMA DE ANÁLISIS RADIOLÓGICO CON MEDGEMMA"
)

print("=" * 80)

print()

print(
    "Modelo:",
    MODEL_ID
)

print(
    "Motor de inferencia:",
    "MedGemma local"
)

print(
    "Interfaz gráfica:",
    "Gradio"
)

print()

print(
    "Opciones disponibles:"
)

print(
    "  [1] Cargar imagen desde el computador"
)

print(
    "  [2] Seleccionar imagen desde el dataset"
)

print()

print(
    "Directorio de resultados:"
)

print(
    OUTPUT_DIR.resolve()
)


if "GRADIO_OUTPUT_DIR" in globals():

    print()

    print(
        "Resultados de Gradio:"
    )

    print(
        GRADIO_OUTPUT_DIR.resolve()
    )


if "dataset_df" in globals():

    print()

    print(
        "Imágenes disponibles en el dataset:",
        len(dataset_df)
    )


print()

print(
    "Acceso:",
    "Local"
)

print(
    "Enlace público:",
    "Desactivado"
)

print()

print(
    "La interfaz se abrirá automáticamente "
    "en el navegador."
)

print()

print("=" * 80)


# ============================================================
# 6. LANZAMIENTO DE LA INTERFAZ
# ============================================================

try:

    interface.launch(

        # ----------------------------------------------------
        # Directorios autorizados
        # ----------------------------------------------------

        allowed_paths=allowed_directories,


        # ----------------------------------------------------
        # Ejecución local
        # ----------------------------------------------------

        share=False,


        # ----------------------------------------------------
        # Abrir automáticamente en navegador
        # ----------------------------------------------------

        inbrowser=True,


        # ----------------------------------------------------
        # Mostrar errores de Gradio
        # ----------------------------------------------------

        show_error=True,


        # ----------------------------------------------------
        # Servidor local
        # ----------------------------------------------------

        server_name="127.0.0.1"

    )


# ============================================================
# 7. DETENCIÓN MANUAL
# ============================================================

except KeyboardInterrupt:

    print()

    print("=" * 80)

    print(
        "INTERFAZ DETENIDA"
    )

    print("=" * 80)


# ============================================================
# 8. CONTROL DE ERRORES
# ============================================================

except Exception as exc:

    print()

    print("=" * 80)

    print(
        "ERROR AL INICIAR LA INTERFAZ"
    )

    print("=" * 80)

    print()

    print(
        "Tipo de error:",
        type(exc).__name__
    )

    print()

    print(
        "Descripción:"
    )

    print(
        str(exc)
    )

# Interpretación final del experimento

**Explicación:** La evaluación combina métricas de funcionamiento, estadísticas del dataset y, cuando existen etiquetas reales, métricas supervisadas.

Las métricas de tiempo, uso de memoria y longitud de respuesta caracterizan el comportamiento computacional del sistema. Las métricas de clasificación solo adquieren significado diagnóstico cuando las etiquetas de referencia proceden de un conjunto validado y el diseño experimental evita sesgos de selección.

La salida de MedGemma debe considerarse experimental. La validación clínica requiere evaluación profesional, un protocolo de referencia y un conjunto de datos correctamente anotado.


# Comparacion Transformers vs llama.cpp

Valores observados experimentalmente en la RTX 2060 (6 GB):

| Aspecto           | Transformers                | llama.cpp               |
| ----------------- | --------------------------- | ----------------------- |
| Modelo base       | MedGemma 1.5 4B             | MedGemma 1.5 4B         |
| Formato           | HF / NF4                    | GGUF Q4_K_M             |
| Runtime           | Transformers 5.16.1 + bnb   | llama.cpp (b10796)      |
| VRAM              | 4.19-4.43 GiB (carga)       | ~5.15 GiB               |
| RAM               | 1.3-2.9 GiB                 | (servidor separado)     |
| Offload           | roto (pesos en `meta`)      | -ngl 999 (GPU)          |
| NaN               | si (fp16) / n/a (fp32)      | no                      |
| Inferencia valida | no                          | si                      |
| Tiempo            | n/a (no infiere)            | ~7 s / 384 tok (~75 tok/s) |
